In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')
def newest(name):
    hits=list(INPUT.rglob(name))
    if not hits: raise FileNotFoundError(f'Missing {name}; attach the successful Day 9.16 notebook output.')
    return max(hits,key=lambda p:(p.stat().st_mtime,str(p)))

def lag1(x):
    x=np.asarray(x,dtype=float)
    if len(x)<3 or np.std(x)==0: return np.nan
    return float(np.corrcoef(x[:-1],x[1:])[0,1])

CASE_PATH=newest('day9_16_case_diagnostics.csv')
PARAM_PATH=newest('day9_16_per_parameter_diagnostics.csv')
case_df=pd.read_csv(CASE_PATH)
param_df=pd.read_csv(PARAM_PATH)
OUT=WORK/('day9-18-posterior-geometry-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
OUT.mkdir(parents=True,exist_ok=False)

summary_rows=[]
corr_rows=[]
plot_paths=[]
for row in [87483,4923]:
    npz_path=newest(f'row_{row}_posterior_by_chain.npz')
    archive=np.load(npz_path)
    # Retain only parameters sampled directly by NUTS; exclude deterministic transforms.
    excluded={'phase','noise_sigma'}
    variables=[]
    for name in archive.files:
        arr=np.asarray(archive[name])
        if name in excluded or arr.ndim<2 or arr.shape[0]!=2 or np.nanstd(arr)==0:
            continue
        arr=arr.reshape(2,-1)
        if np.all(np.isfinite(arr)):
            variables.append((name,arr))
    if len(variables)<2: raise ValueError(f'Not enough varying sampled parameters for row {row}: {[x[0] for x in variables]}')
    available=param_df[param_df.simulation_row.astype(int).eq(row)].set_index('parameter')
    for name,arr in variables:
        pooled=np.concatenate(arr)
        sd=float(np.std(pooled,ddof=1))
        denom=sd if sd>0 else np.nan
        rh=float(available.loc[name,'r_hat']) if name in available.index else np.nan
        ess=float(available.loc[name,'effective_sample_size']) if name in available.index else np.nan
        summary_rows.append({'simulation_row':row,'parameter':name,
            'chain0_mean':float(np.mean(arr[0])),'chain1_mean':float(np.mean(arr[1])),
            'pooled_sd':sd,'standardized_chain_mean_gap':float(abs(np.mean(arr[0])-np.mean(arr[1]))/denom) if np.isfinite(denom) else np.nan,
            'chain0_lag1_autocorrelation':lag1(arr[0]),'chain1_lag1_autocorrelation':lag1(arr[1]),
            'r_hat':rh,'effective_sample_size':ess})
    names=[x[0] for x in variables]
    flat=pd.DataFrame({name:arr.reshape(-1) for name,arr in variables})
    corr=flat.corr(method='spearman')
    for i,a in enumerate(names):
        for b in names[i+1:]:
            rho=float(corr.loc[a,b])
            corr_rows.append({'simulation_row':row,'parameter_a':a,'parameter_b':b,'spearman_rho':rho,'abs_rho':abs(rho)})
    # One row per case: traces by parameter and correlation heatmap.
    n=len(variables)
    fig,axes=plt.subplots(2,n,figsize=(max(12,3*n),7),squeeze=False,constrained_layout=True)
    for j,(name,arr) in enumerate(variables):
        axes[0,j].plot(arr[0],lw=.55,label='chain 0')
        axes[0,j].plot(arr[1],lw=.55,alpha=.8,label='chain 1')
        axes[0,j].set_title(name,fontsize=9)
        axes[0,j].set_xlabel('draw')
        axes[1,j].axis('off')
    # Put the full correlation heatmap in the bottom row's first available axis and hide the rest.
    ax=axes[1,0]
    ax.clear()
    im=ax.imshow(corr.to_numpy(),vmin=-1,vmax=1,cmap='coolwarm',aspect='auto')
    ax.set_xticks(range(len(names)),names,rotation=65,ha='right',fontsize=8)
    ax.set_yticks(range(len(names)),names,fontsize=8)
    ax.set_title('Spearman correlation of pooled draws',fontsize=9)
    for j in range(1,n): axes[1,j].axis('off')
    axes[0,0].legend(fontsize=8)
    fig.colorbar(im,ax=ax,fraction=.046,pad=.04)
    fig.suptitle(f'Day 9.16 posterior geometry — row {row}',fontsize=13)
    image=OUT/f'day9_18_row_{row}_trace_correlation.png'
    fig.savefig(image,dpi=140)
    plt.close(fig)
    plot_paths.append(image.name)

summary_df=pd.DataFrame(summary_rows).sort_values(['simulation_row','parameter'])
corr_df=pd.DataFrame(corr_rows).sort_values(['simulation_row','abs_rho'],ascending=[True,False])
summary_df.to_csv(OUT/'day9_18_chain_geometry_summary.csv',index=False)
corr_df.to_csv(OUT/'day9_18_parameter_correlations.csv',index=False)
case_review=case_df[case_df.simulation_row.astype(int).isin([87483,4923])].copy()
case_review.to_csv(OUT/'day9_18_source_case_diagnostics.csv',index=False)

report={'created_utc':datetime.now(timezone.utc).isoformat(),'source_case_diagnostics':str(CASE_PATH),
    'source_parameter_diagnostics':str(PARAM_PATH),'rows':[87483,4923],
    'purpose':'Post-hoc chain and correlation inspection only; no new sampling or posterior validation.',
    'notes':['Strong pooled parameter correlation can indicate a ridge or weak identifiability, but correlation alone does not prove the cause of poor mixing.',
        'The saved chains for row 87483 and row 4923 already failed Day 9.16 sampler screening; these plots do not change those statuses.',
        'Row 87483 fixes duration and both rows fix period, so uncertainty in those fixed values is not represented.',
        'Synthetic conditional diagnostics are not calibration or real-star validation.'],
    'outputs':['day9_18_chain_geometry_summary.csv','day9_18_parameter_correlations.csv','day9_18_source_case_diagnostics.csv']+plot_paths}
(OUT/'day9_18_geometry_summary.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print(summary_df.to_string(index=False))
print('\nLargest absolute parameter correlations:')
print(corr_df.groupby('simulation_row').head(8).to_string(index=False))
print('\nSaved diagnostics:',OUT)
print(json.dumps(report,indent=2))